## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Core imports
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
from tqdm.notebook import tqdm

# Project imports
from src.data import PotatoDataModule
from src.data.datasets import PotatoDiseaseDataset
from src.data.transforms import get_train_transforms, get_val_transforms, AndeanFieldAugmentation
from src.models import create_model
from src.utils.metrics import compute_accuracy, compute_f1

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

In [ ]:
# Paths
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "plantvillage"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print(f"Data directory: {DATA_DIR}")
print(f"Exists: {DATA_DIR.exists()}")
print(f"\nAvailable classes:")
if DATA_DIR.exists():
    for folder in sorted(DATA_DIR.iterdir()):
        if folder.is_dir():
            n_images = len(list(folder.glob("*.[jJ][pP][gG]")) + list(folder.glob("*.png")))
            print(f"  {folder.name}: {n_images} images")

## 2. Exploración del Dataset

In [ ]:
# Load dataset without transforms to see raw images
dataset_raw = PotatoDiseaseDataset(
    root=str(DATA_DIR),
    transform=None,
    class_filter="Potato",  # Only potato classes
)

print(f"Total samples: {len(dataset_raw)}")
print(f"Classes: {dataset_raw.classes}")
print(f"Class mapping: {dataset_raw.class_to_idx}")

In [ ]:
# Class distribution
labels = [sample[1] for sample in dataset_raw.samples]
class_counts = Counter(labels)

# Map to class names
idx_to_class = {v: k for k, v in dataset_raw.class_to_idx.items()}
class_distribution = {idx_to_class[idx]: count for idx, count in class_counts.items()}

print("\nClass Distribution:")
for cls, count in sorted(class_distribution.items()):
    pct = count / len(dataset_raw) * 100
    print(f"  {cls}: {count} ({pct:.1f}%)")

In [ ]:
# Visualize distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
classes = list(class_distribution.keys())
counts = list(class_distribution.values())
colors = sns.color_palette("husl", len(classes))

axes[0].barh(classes, counts, color=colors)
axes[0].set_xlabel('Number of Samples')
axes[0].set_title('Class Distribution')
for i, (cls, count) in enumerate(zip(classes, counts)):
    axes[0].text(count + 20, i, str(count), va='center')

# Pie chart
axes[1].pie(counts, labels=classes, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.show()

## 3. Visualización de Muestras

In [ ]:
def show_samples(dataset, n_per_class=3, figsize=(15, 10)):
    """Display sample images from each class."""
    classes = dataset.classes
    n_classes = len(classes)
    
    fig, axes = plt.subplots(n_classes, n_per_class, figsize=figsize)
    
    # Group samples by class
    class_samples = {i: [] for i in range(n_classes)}
    for path, label in dataset.samples:
        class_samples[label].append(path)
    
    for class_idx, class_name in enumerate(classes):
        samples = class_samples[class_idx][:n_per_class]
        
        for sample_idx, img_path in enumerate(samples):
            ax = axes[class_idx, sample_idx] if n_classes > 1 else axes[sample_idx]
            
            img = Image.open(img_path)
            ax.imshow(img)
            ax.axis('off')
            
            if sample_idx == 0:
                ax.set_ylabel(class_name.replace('_', ' ').title(), 
                             fontsize=12, rotation=0, ha='right', va='center')
    
    plt.suptitle('Sample Images by Class', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

show_samples(dataset_raw, n_per_class=4)

In [ ]:
# Image statistics
def analyze_image_stats(dataset, n_samples=100):
    """Analyze image dimensions and statistics."""
    widths, heights = [], []
    means, stds = [], []
    
    indices = np.random.choice(len(dataset), min(n_samples, len(dataset)), replace=False)
    
    for idx in tqdm(indices, desc="Analyzing images"):
        img_path = dataset.samples[idx][0]
        img = Image.open(img_path).convert('RGB')
        
        widths.append(img.width)
        heights.append(img.height)
        
        img_array = np.array(img) / 255.0
        means.append(img_array.mean(axis=(0, 1)))
        stds.append(img_array.std(axis=(0, 1)))
    
    print(f"\nImage Dimensions:")
    print(f"  Width: {np.min(widths)} - {np.max(widths)} (mean: {np.mean(widths):.0f})")
    print(f"  Height: {np.min(heights)} - {np.max(heights)} (mean: {np.mean(heights):.0f})")
    
    print(f"\nChannel Statistics (R, G, B):")
    print(f"  Mean: {np.mean(means, axis=0)}")
    print(f"  Std: {np.mean(stds, axis=0)}")
    
    return widths, heights

widths, heights = analyze_image_stats(dataset_raw, n_samples=200)

## 4. Análisis de Augmentaciones

In [ ]:
def visualize_augmentations(image_path, n_augmented=6):
    """Show original image and augmented versions."""
    from torchvision import transforms
    
    img = Image.open(image_path).convert('RGB')
    
    # Define transforms
    train_transform = get_train_transforms(image_size=224, strength='medium')
    andean_aug = AndeanFieldAugmentation(p=0.8, intensity='medium')
    
    # Combined Andean transform
    andean_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        andean_aug,
    ])
    
    fig, axes = plt.subplots(2, n_augmented + 1, figsize=(18, 7))
    
    # Original
    axes[0, 0].imshow(img)
    axes[0, 0].set_title('Original', fontsize=10)
    axes[0, 0].axis('off')
    axes[1, 0].imshow(img)
    axes[1, 0].set_title('Original', fontsize=10)
    axes[1, 0].axis('off')
    
    # Standard augmentations
    for i in range(n_augmented):
        aug_img = train_transform(img)
        # Denormalize for visualization
        aug_img = aug_img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + \
                  torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        aug_img = aug_img.permute(1, 2, 0).numpy().clip(0, 1)
        
        axes[0, i + 1].imshow(aug_img)
        axes[0, i + 1].set_title(f'Standard #{i+1}', fontsize=10)
        axes[0, i + 1].axis('off')
    
    # Andean augmentations
    for i in range(n_augmented):
        aug_img = andean_transform(img)
        aug_img = aug_img.permute(1, 2, 0).numpy().clip(0, 1)
        
        axes[1, i + 1].imshow(aug_img)
        axes[1, i + 1].set_title(f'Andean #{i+1}', fontsize=10)
        axes[1, i + 1].axis('off')
    
    axes[0, 0].set_ylabel('Standard\nAugmentations', fontsize=11)
    axes[1, 0].set_ylabel('Andean Field\nAugmentations', fontsize=11)
    
    plt.suptitle('Data Augmentation Comparison', fontsize=14)
    plt.tight_layout()
    plt.show()

# Pick a sample image
sample_path = dataset_raw.samples[0][0]
visualize_augmentations(sample_path)

## 5. Cargar y Evaluar Modelos

In [ ]:
# List available checkpoints
print("Available experiments:")
if OUTPUT_DIR.exists():
    for exp_dir in sorted(OUTPUT_DIR.iterdir()):
        if exp_dir.is_dir():
            checkpoints = list(exp_dir.glob("*.pt"))
            print(f"  {exp_dir.name}/")
            for ckpt in checkpoints:
                print(f"    └── {ckpt.name}")
else:
    print("  No experiments found. Run training first!")

In [ ]:
def load_model_from_checkpoint(checkpoint_path, backbone='mobilenet_v3_small', num_classes=3):
    """Load a trained model from checkpoint."""
    model = create_model(
        model_type='baseline',
        backbone=backbone,
        num_classes=num_classes,
        pretrained=False,
    )
    
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(DEVICE)
    model.eval()
    
    print(f"Loaded model from: {checkpoint_path}")
    print(f"Epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"Metrics: {checkpoint.get('metrics', {})}")
    
    return model, checkpoint

# Load best model (update path as needed)
# Find most recent experiment
exp_dirs = sorted([d for d in OUTPUT_DIR.iterdir() if d.is_dir()], reverse=True)
if exp_dirs:
    latest_exp = exp_dirs[0]
    best_model_path = latest_exp / "best_model.pt"
    
    if best_model_path.exists():
        model, checkpoint = load_model_from_checkpoint(best_model_path)
    else:
        print(f"No best_model.pt found in {latest_exp}")
else:
    print("No experiments found. Train a model first!")

In [ ]:
# Create test dataloader
val_transform = get_val_transforms(image_size=224)

dataset_val = PotatoDiseaseDataset(
    root=str(DATA_DIR),
    transform=val_transform,
    class_filter="Potato",
)

# Use a subset for quick evaluation
from torch.utils.data import DataLoader, Subset

eval_indices = np.random.choice(len(dataset_val), min(500, len(dataset_val)), replace=False)
eval_subset = Subset(dataset_val, eval_indices)
eval_loader = DataLoader(eval_subset, batch_size=32, shuffle=False, num_workers=2)

print(f"Evaluation samples: {len(eval_subset)}")

In [ ]:
@torch.no_grad()
def evaluate_model(model, dataloader, classes):
    """Evaluate model and return predictions."""
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    for images, labels in tqdm(dataloader, desc="Evaluating"):
        images = images.to(DEVICE)
        
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    accuracy = (all_preds == all_labels).mean()
    print(f"\nAccuracy: {accuracy:.4f}")
    
    return all_preds, all_labels, all_probs

# Run evaluation
if 'model' in dir():
    preds, labels, probs = evaluate_model(model, eval_loader, dataset_val.classes)

## 6. Visualización de Resultados

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

def plot_confusion_matrix(y_true, y_pred, classes, normalize=True):
    """Plot confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='.2f' if normalize else 'd',
                xticklabels=classes, yticklabels=classes,
                cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix' + (' (Normalized)' if normalize else ''))
    plt.tight_layout()
    plt.show()

if 'preds' in dir():
    plot_confusion_matrix(labels, preds, dataset_val.classes)
    
    print("\nClassification Report:")
    print(classification_report(labels, preds, target_names=dataset_val.classes))

In [ ]:
def plot_confidence_distribution(probs, preds, labels):
    """Plot confidence distribution for correct and incorrect predictions."""
    correct_mask = preds == labels
    
    # Get max probabilities (confidence)
    confidences = probs.max(axis=1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Overall distribution
    axes[0].hist(confidences[correct_mask], bins=30, alpha=0.7, 
                 label=f'Correct ({correct_mask.sum()})', color='green')
    axes[0].hist(confidences[~correct_mask], bins=30, alpha=0.7,
                 label=f'Incorrect ({(~correct_mask).sum()})', color='red')
    axes[0].set_xlabel('Confidence')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Confidence Distribution')
    axes[0].legend()
    
    # Box plot by class
    data_for_box = []
    labels_for_box = []
    for is_correct, name in [(True, 'Correct'), (False, 'Incorrect')]:
        mask = correct_mask == is_correct
        data_for_box.extend(confidences[mask])
        labels_for_box.extend([name] * mask.sum())
    
    import pandas as pd
    df = pd.DataFrame({'Confidence': data_for_box, 'Prediction': labels_for_box})
    sns.boxplot(data=df, x='Prediction', y='Confidence', ax=axes[1])
    axes[1].set_title('Confidence by Prediction Correctness')
    
    plt.tight_layout()
    plt.show()

if 'probs' in dir():
    plot_confidence_distribution(probs, preds, labels)

## 7. Análisis de Errores

In [ ]:
def show_misclassified(dataset, indices, preds, labels, probs, classes, n_show=12):
    """Display misclassified samples."""
    # Find misclassified
    incorrect_mask = preds != labels
    incorrect_indices = np.where(incorrect_mask)[0]
    
    if len(incorrect_indices) == 0:
        print("No misclassified samples!")
        return
    
    n_show = min(n_show, len(incorrect_indices))
    show_indices = incorrect_indices[:n_show]
    
    cols = 4
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = axes.flatten()
    
    for i, idx in enumerate(show_indices):
        # Get original image
        dataset_idx = indices[idx]
        img_path = dataset.samples[dataset_idx][0]
        img = Image.open(img_path)
        
        true_class = classes[labels[idx]]
        pred_class = classes[preds[idx]]
        confidence = probs[idx].max() * 100
        
        axes[i].imshow(img)
        axes[i].set_title(
            f"True: {true_class}\nPred: {pred_class} ({confidence:.1f}%)",
            color='red'
        )
        axes[i].axis('off')
    
    # Hide empty axes
    for i in range(n_show, len(axes)):
        axes[i].axis('off')
    
    plt.suptitle('Misclassified Samples', fontsize=14)
    plt.tight_layout()
    plt.show()

if 'preds' in dir():
    show_misclassified(dataset_val, eval_indices, preds, labels, probs, dataset_val.classes)

In [ ]:
def analyze_errors_by_class(preds, labels, classes):
    """Analyze error patterns."""
    incorrect_mask = preds != labels
    
    print("Error Analysis by Class:")
    print("=" * 50)
    
    for class_idx, class_name in enumerate(classes):
        class_mask = labels == class_idx
        class_incorrect = incorrect_mask & class_mask
        
        n_total = class_mask.sum()
        n_incorrect = class_incorrect.sum()
        
        print(f"\n{class_name}:")
        print(f"  Total: {n_total}, Incorrect: {n_incorrect} ({n_incorrect/n_total*100:.1f}%)")
        
        if n_incorrect > 0:
            # What were they predicted as?
            wrong_preds = preds[class_incorrect]
            pred_counts = Counter(wrong_preds)
            print(f"  Misclassified as:")
            for pred_idx, count in pred_counts.most_common():
                print(f"    {classes[pred_idx]}: {count}")

if 'preds' in dir():
    analyze_errors_by_class(preds, labels, dataset_val.classes)

## 8. Predicción en Imágenes Individuales

In [ ]:
@torch.no_grad()
def predict_image(model, image_path, transform, classes, show=True):
    """Predict class for a single image."""
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(DEVICE)
    
    model.eval()
    logits = model(img_tensor)
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = probs.argmax().item()
    
    if show:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Show image
        axes[0].imshow(img)
        axes[0].set_title(f'Prediction: {classes[pred_idx]}\nConfidence: {probs[pred_idx]*100:.1f}%')
        axes[0].axis('off')
        
        # Show probabilities
        probs_np = probs.cpu().numpy()
        colors = ['green' if i == pred_idx else 'steelblue' for i in range(len(classes))]
        bars = axes[1].barh(classes, probs_np, color=colors)
        axes[1].set_xlim(0, 1)
        axes[1].set_xlabel('Probability')
        axes[1].set_title('Class Probabilities')
        
        for bar, prob in zip(bars, probs_np):
            axes[1].text(prob + 0.02, bar.get_y() + bar.get_height()/2, 
                        f'{prob*100:.1f}%', va='center')
        
        plt.tight_layout()
        plt.show()
    
    return classes[pred_idx], probs_np

# Test on a random image
if 'model' in dir():
    random_idx = np.random.randint(len(dataset_val))
    random_path = dataset_val.samples[random_idx][0]
    true_label = dataset_val.classes[dataset_val.samples[random_idx][1]]
    
    print(f"True label: {true_label}")
    predicted, probs = predict_image(model, random_path, val_transform, dataset_val.classes)

## 9. Comparar Múltiples Experimentos

In [ ]:
def load_training_history(exp_dir):
    """Load training history from experiment."""
    history_path = exp_dir / "history.pt"
    if history_path.exists():
        return torch.load(history_path)
    return None

def plot_experiment_comparison(output_dir, metric='accuracy'):
    """Compare training curves across experiments."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for exp_dir in sorted(output_dir.iterdir()):
        if not exp_dir.is_dir():
            continue
            
        history = load_training_history(exp_dir)
        if history is None:
            continue
        
        exp_name = exp_dir.name[:30]  # Truncate long names
        
        # Training curve
        train_metric = [ep.get(metric, ep.get('loss')) for ep in history['train']]
        val_metric = [ep.get(metric, ep.get('loss')) for ep in history['val']]
        
        epochs = range(1, len(train_metric) + 1)
        
        axes[0].plot(epochs, train_metric, label=f'{exp_name} (train)')
        axes[1].plot(epochs, val_metric, label=f'{exp_name} (val)')
    
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel(metric.title())
    axes[0].set_title(f'Training {metric.title()}')
    axes[0].legend(fontsize=8)
    
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel(metric.title())
    axes[1].set_title(f'Validation {metric.title()}')
    axes[1].legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()

if OUTPUT_DIR.exists():
    plot_experiment_comparison(OUTPUT_DIR, metric='accuracy')
    plot_experiment_comparison(OUTPUT_DIR, metric='loss')

---

## 📝 Notas

- **TensorBoard**: Para visualización más detallada, ejecuta `tensorboard --logdir outputs/` en terminal
- **Nuevos experimentos**: Actualiza las rutas de checkpoints al entrenar nuevos modelos
- **OOD Detection**: Para detección de clases fuera de distribución, ver `src/utils/ood_detection.py`